# Round 3 EDA — HYDROGEL_PACK, VELVETFRUIT_EXTRACT, VEV options

Three days of historical data:
- day 0 → TTE = 8d at start
- day 1 → TTE = 7d at start
- day 2 → TTE = 6d at start
- (live submission day 3 → TTE = 5d at start)

10000 ticks/day, ts in steps of 100.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

BASE = '/Users/markiejr/Propserity_4/data/ROUND3/ROUND_3'
frames = []
for d in [0, 1, 2]:
    df = pd.read_csv(f'{BASE}/prices_round_3_day_{d}.csv', sep=';')
    df['day'] = d
    frames.append(df)
px = pd.concat(frames, ignore_index=True)
px['t_global'] = px['day'] * 1_000_000 + px['timestamp']
px = px.sort_values('t_global').reset_index(drop=True)

# Pivot to wide form for mid prices
mid = px.pivot_table(index='t_global', columns='product', values='mid_price', aggfunc='first')
print('shape:', mid.shape)
mid.head()

## 1. Delta-1 products: HYDROGEL_PACK and VELVETFRUIT_EXTRACT

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax[0].plot(mid.index, mid['HYDROGEL_PACK'], lw=0.5)
ax[0].axhline(10000, color='r', ls='--', alpha=0.5, label='10000 anchor')
ax[0].set_title('HYDROGEL_PACK mid (3 days)')
ax[0].legend()
ax[1].plot(mid.index, mid['VELVETFRUIT_EXTRACT'], lw=0.5, color='purple')
ax[1].set_title('VELVETFRUIT_EXTRACT mid (3 days)')
for d in [1, 2]:
    for a in ax:
        a.axvline(d * 1_000_000, color='k', ls=':', alpha=0.3)
plt.tight_layout(); plt.show()

for prod in ['HYDROGEL_PACK', 'VELVETFRUIT_EXTRACT']:
    s = mid[prod].dropna()
    rets = s.diff().dropna()
    print(f'{prod}: mean={s.mean():.2f} std={s.std():.2f} ret_std={rets.std():.3f} '
          f'lag1_ACF={rets.autocorr(1):+.3f}')

**Reading:** HYDROGEL_PACK is strongly anchored at 10,000 (stationary, MR). VELVETFRUIT_EXTRACT drifts upward (5244 → 5265 → 5295 over 3 days) and has narrow spread (~5).

## 2. VEV vouchers: parity check

For deep ITM calls, voucher price ≈ S − K. Time value is voucher_mid − max(0, S − K).

In [ ]:
strikes = [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500]
S = mid['VELVETFRUIT_EXTRACT']
rows = []
for K in strikes:
    C = mid[f'VEV_{K}']
    intrinsic = (S - K).clip(lower=0)
    tv = (C - intrinsic).dropna()
    rows.append({'K': K, 'avg_C': C.mean(), 'avg_intrinsic': intrinsic.mean(),
                 'avg_TV': tv.mean(), 'std_TV': tv.std()})
pd.DataFrame(rows).round(2)

Deep ITM (4000, 4500) trade at exact intrinsic — TV ≈ 0. Deep OTM (6000, 6500) are pinned at 0.50 floor regardless of model fair (TV looks large because S−K is negative for them and their mid is floored).

## 3. Implied volatility surface (Black-Scholes, r=0)

In [ ]:
from math import erf, sqrt, log
def norm_cdf(x): return 0.5*(1+erf(x/sqrt(2)))
def bs_call(S, K, T, sigma):
    if T<=0 or sigma<=0: return max(0.0, S-K)
    d1 = (log(S/K)+0.5*sigma*sigma*T)/(sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S*norm_cdf(d1) - K*norm_cdf(d2)
def implied_vol(C, S, K, T, lo=1e-4, hi=5.0):
    if C <= max(0, S-K) + 1e-9: return 0.0
    if C >= S: return float('nan')
    for _ in range(80):
        m = 0.5*(lo+hi)
        if bs_call(S,K,T,m) > C: hi = m
        else: lo = m
    return 0.5*(lo+hi)

# IV at end-of-day snapshots
iv_rows = []
for d in [0, 1, 2]:
    for ts in [0, 250_000, 500_000, 750_000, 999_900]:
        tg = d * 1_000_000 + ts
        if tg not in mid.index: continue
        s_now = mid.loc[tg, 'VELVETFRUIT_EXTRACT']
        TTE = (8 - d) - ts/1_000_000
        for K in strikes:
            c = mid.loc[tg, f'VEV_{K}']
            if pd.isna(c): continue
            iv = implied_vol(c, s_now, K, TTE)
            iv_rows.append({'day': d, 'ts': ts, 'TTE': TTE, 'S': s_now, 'K': K, 'C': c, 'IV': iv})
iv_df = pd.DataFrame(iv_rows)
iv_df['moneyness'] = iv_df['K'] / iv_df['S']
iv_df.head()

In [ ]:
# IV smile per day
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
for d in [0, 1, 2]:
    sub = iv_df[(iv_df['day']==d) & (iv_df['ts']==500_000)]
    ax.plot(sub['moneyness'], sub['IV'], 'o-', label=f'day {d}')
ax.set_xlabel('K/S (moneyness)'); ax.set_ylabel('IV (per √day)')
ax.set_title('IV smile — note 6000/6500 are floor-pinned, treat as garbage')
ax.legend(); ax.grid(alpha=0.3); plt.show()

# Average IV per strike (excluding pinned 6000/6500)
smile = iv_df[iv_df['K'].between(5000, 5500)].groupby('K')['IV'].agg(['mean','std','count'])
print(smile)

**Smile reading:** strikes 5000–5500 cluster around IV ≈ 0.012–0.013 per √day. The 6000/6500 'IV' is artifact of the 0.50 floor (vouchers can't trade below 0.50 even though BS says they're worth ~0). Use a flat σ ≈ 0.0125 per √day for fair-value computations on strikes 5000–5500.

## 4. Realized vs implied vol (mean-reversion adjustment)

In [ ]:
S_series = mid['VELVETFRUIT_EXTRACT'].dropna()
log_ret_tick = np.log(S_series).diff().dropna()
# Tick vol -> per sqrt-day
tick_vol = log_ret_tick.std()
vol_per_sqrt_day_naive = tick_vol * np.sqrt(10000)
print(f'tick log-ret std: {tick_vol:.6f}')
print(f'naive scaled per-sqrt-day vol: {vol_per_sqrt_day_naive:.4f}')
print(f'lag-1 ACF: {log_ret_tick.autocorr(1):.4f}')

# Variance-ratio test: var(N-tick) / N / var(1-tick)
for lag in [1, 5, 10, 50, 100, 500, 1000, 5000]:
    ret = np.log(S_series).diff(lag).dropna()
    vr = ret.var() / (lag * log_ret_tick.var())
    print(f'  lag={lag:5d}: var-ratio={vr:.3f}  (1=GBM, <1=MR, >1=trend)')

**Reading:** if the variance ratio drops below 1 at long lags, the underlying is mean-reverting and 'true' option-relevant vol is *lower* than the naive tick-scaled estimate. Implied vol of 0.0125/√day reflects that — market is pricing the MR. Don't try to short vol naively.

## 5. Strategy summary going into the algo

1. **HYDROGEL_PACK**: MM around 10,000 anchor. Reuse OSM-style (Kalman + ACF) framework from R2, but spread is ~16 — quotes at ±5 to ±7.
2. **VELVETFRUIT_EXTRACT**: tight MM (offsets ±1–2). Spread is ~5 so margins are thin; need mostly micro-price + small ACF bias. Watch for upward drift.
3. **VEV_4000, VEV_4500** (deep ITM, pure intrinsic): MM around `S − K`. Spreads are wider (~16–21) so good margin. Effectively a delta-1 synthetic.
4. **VEV_5000–5500** (active options): MM around BS fair with σ=0.0125/√day. Take when ask < BS_fair − 1 or bid > BS_fair + 1.
5. **VEV_6000, VEV_6500**: skip (pinned at 0.50 floor, no edge).
6. **Delta hedge**: track aggregate option delta from voucher fills, neutralize via VELVETFRUIT_EXTRACT if |Δ| exceeds 30.